# Lampiran — Kode Scraping Ulasan Tokopedia
Produk **SH-RD Protein Cream 50ml** · keluaran `tokopedia_reviews.csv`

Jalankan keempat sel di bawah ini secara berurutan pada Google Colaboratory.

In [ ]:
# ---------------------------------------------------------------------
# BAGIAN 1 - INSTALASI GOOGLE CHROME DAN PUSTAKA PENDUKUNG
# ---------------------------------------------------------------------
import os
import subprocess

def jalankan(perintah, label):
    """Menjalankan perintah shell dan menampilkan galat bila gagal."""
    hasil = subprocess.run(perintah, capture_output=True, text=True)
    if hasil.returncode != 0:
        print(f'  {label} gagal (exit {hasil.returncode})')
        print(f'  {hasil.stderr[-500:]}')
    return hasil

URL_CHROME = ('https://dl.google.com/linux/direct/'
              'google-chrome-stable_current_amd64.deb')
BERKAS_DEB = 'google-chrome-stable_current_amd64.deb'

print('Tahap 1/5 - Mengunduh Google Chrome')
jalankan(['wget', '-q', '-O', BERKAS_DEB, URL_CHROME], 'Unduh Chrome')

print('Tahap 2/5 - Memperbarui daftar paket')
jalankan(['apt-get', 'update', '-qq'], 'apt update')

print('Tahap 3/5 - Memasang Google Chrome')
jalankan(['apt-get', 'install', '-y', f'./{BERKAS_DEB}'], 'Pasang Chrome')
jalankan(['apt-get', 'install', '-f', '-y'], 'Perbaiki dependensi')

print('Tahap 4/5 - Mencari lokasi binary Chrome')
CHROME_PATH = None
for nama in ['google-chrome-stable', 'google-chrome', 'chromium-browser']:
    hasil = subprocess.run(['which', nama], capture_output=True, text=True)
    if hasil.stdout.strip():
        CHROME_PATH = hasil.stdout.strip()
        break
if not CHROME_PATH:
    for lokasi in ['/usr/bin/google-chrome-stable',
                   '/usr/bin/google-chrome',
                   '/opt/google/chrome/google-chrome']:
        if os.path.exists(lokasi):
            CHROME_PATH = lokasi
            break
if not CHROME_PATH:
    raise RuntimeError('Google Chrome tidak berhasil dipasang.')

os.environ['CHROME_BIN_PATH'] = CHROME_PATH
with open('/tmp/chrome_path.txt', 'w') as berkas:
    berkas.write(CHROME_PATH)
print(f'  Chrome ditemukan pada: {CHROME_PATH}')

print('Tahap 5/5 - Memasang pustaka Python')
jalankan(['pip', 'install', '-q', 'selenium', 'webdriver-manager',
          'beautifulsoup4', 'pandas', 'lxml'], 'Pasang pustaka')

import bs4
import pandas as pd
import selenium

print(f'Selenium       : {selenium.__version__}')
print(f'BeautifulSoup4 : {bs4.__version__}')
print(f'Pandas         : {pd.__version__}')

In [ ]:
# ---------------------------------------------------------------------
# BAGIAN 2 - KONFIGURASI DRIVER DAN VERIFIKASI SELEKTOR
# ---------------------------------------------------------------------
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

CHROME_PATH = os.environ.get('CHROME_BIN_PATH')
if not CHROME_PATH and os.path.exists('/tmp/chrome_path.txt'):
    CHROME_PATH = open('/tmp/chrome_path.txt').read().strip()

URL_ULASAN = (
    'https://www.tokopedia.com/shrd/'
    'sh-rd-protein-cream-50ml-hair-moisturizer-heat-protector-'
    'catokan-kering-perawatan-haircare-leave-on-conditioner-'
    'melembutkan-1729956044502370862'
)
JUMLAH_DATA = 5000
NAMA_AGEN = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
             'AppleWebKit/537.36 (KHTML, like Gecko) '
             'Chrome/124.0.0.0 Safari/537.36')

def buat_driver():
    """Menyiapkan peramban Chrome mode headless untuk proses scraping."""
    opsi = Options()
    opsi.add_argument('--headless=new')
    opsi.add_argument('--no-sandbox')
    opsi.add_argument('--disable-dev-shm-usage')
    opsi.add_argument('--disable-gpu')
    opsi.add_argument('--window-size=1920,1080')
    opsi.add_argument('--disable-blink-features=AutomationControlled')
    opsi.add_argument('--lang=id-ID')
    opsi.add_argument(f'user-agent={NAMA_AGEN}')
    opsi.add_experimental_option('excludeSwitches', ['enable-automation'])
    opsi.add_experimental_option('useAutomationExtension', False)
    opsi.binary_location = CHROME_PATH
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=opsi)

# Daftar selektor kandidat, diperiksa berurutan hingga ditemukan
SELEKTOR = [
    ('article', {'class': 'css-ccpe8t'}),
    ('article', {'data-testid': 'review-item'}),
    ('div', {'data-testid': 'review-item'}),
    ('div', {'class': 'css-ccpe8t'}),
]

driver = buat_driver()
driver.get(URL_ULASAN)
time.sleep(6)
driver.execute_script('window.scrollTo(0, 5000)')
time.sleep(3)

sup = BeautifulSoup(driver.page_source, 'html.parser')
for tag, atribut in SELEKTOR:
    ditemukan = sup.findAll(tag, attrs=atribut)
    status = f'{len(ditemukan)} ulasan' if ditemukan else 'tidak ditemukan'
    print(f'Selektor {tag} {atribut} : {status}')
print(f'Total elemen <article> pada halaman: {len(sup.findAll("article"))}')
driver.quit()

In [ ]:
# ---------------------------------------------------------------------
# BAGIAN 3 - PROSES SCRAPING ULASAN
# ---------------------------------------------------------------------
import re
from datetime import datetime, timedelta

from selenium.webdriver.common.by import By

PETA_RATING = {
    'bintang 5': 5, 'bintang 4': 4, 'bintang 3': 3,
    'bintang 2': 2, 'bintang 1': 1,
}
KATA_WAKTU = ['lalu', 'hari', 'bulan', 'tahun', 'minggu', 'jam',
              'baru', 'tadi']

def ambil_tanggal(wadah):
    """Mengambil tanggal relatif ulasan dengan empat strategi berurutan.

    Tokopedia menuliskan tanggal dalam bentuk relatif, misalnya
    '2 hari lalu' atau '3 bulan lalu', dan struktur elemennya dapat
    berbeda antar halaman sehingga diperlukan beberapa alternatif.
    """
    # Strategi 1: atribut data-unify bernilai Typography
    for elemen in wadah.find_all('p', {'data-unify': 'Typography'}):
        teks = elemen.get_text(strip=True)
        if any(k in teks.lower() for k in KATA_WAKTU) and len(teks) < 30:
            return teks

    # Strategi 2: kelas CSS spesifik hasil pemeriksaan DevTools
    for elemen in wadah.find_all('p'):
        kelas = ' '.join(elemen.get('class', []))
        if 'e1qvo2ff8' in kelas:
            teks = elemen.get_text(strip=True)
            if any(k in teks.lower() for k in KATA_WAKTU):
                return teks

    # Strategi 3: pencocokan pola 'angka satuan lalu'
    for elemen in wadah.find_all(['p', 'span']):
        teks = elemen.get_text(strip=True)
        if re.search(r'\d+\s*(jam|hari|minggu|bulan|tahun)\s*lalu',
                     teks, re.I):
            return teks

    # Strategi 4: kata 'lalu' pada teks pendek
    for elemen in wadah.find_all(['p', 'span', 'div']):
        teks = elemen.get_text(strip=True)
        if 'lalu' in teks.lower() and len(teks) < 25:
            return teks

    return ''

def konversi_tanggal(teks):
    """Mengubah tanggal relatif menjadi objek datetime perkiraan."""
    if not teks:
        return pd.NaT
    teks = teks.lower().strip()
    waktu_kini = datetime.now()
    angka = re.search(r'(\d+)', teks)
    n = int(angka.group(1)) if angka else 1
    if 'jam' in teks:
        return waktu_kini - timedelta(hours=n)
    if 'hari' in teks:
        return waktu_kini - timedelta(days=n)
    if 'minggu' in teks:
        return waktu_kini - timedelta(weeks=n)
    if 'bulan' in teks:
        return waktu_kini - timedelta(days=n * 30)
    if 'tahun' in teks:
        return waktu_kini - timedelta(days=n * 365)
    if 'baru' in teks or 'tadi' in teks:
        return waktu_kini
    return pd.NaT

def ambil_wadah_ulasan(sup):
    """Mengembalikan daftar elemen ulasan pada satu halaman."""
    for tag, atribut in SELEKTOR:
        wadah = sup.findAll(tag, attrs=atribut)
        if wadah:
            return wadah
    return sup.findAll('article')

def ambil_teks(wadah, kandidat, cadangan):
    """Mengambil teks elemen pertama yang cocok dari daftar kandidat."""
    for tag, atribut in kandidat:
        elemen = wadah.find(tag, atribut)
        if elemen:
            return elemen.text.strip()
    return cadangan

def scraping_ulasan(url, jumlah_data, batas_halaman=200):
    """Mengumpulkan ulasan produk dari halaman Tokopedia.

    Argumen:
        url           : alamat halaman produk
        jumlah_data   : jumlah ulasan maksimum yang dikumpulkan
        batas_halaman : batas halaman untuk mencegah perulangan tanpa henti

    Mengembalikan DataFrame berisi platform, username, rating,
    tanggal_raw, tanggal, dan content.
    """
    driver = buat_driver()
    print(f'Membuka halaman: {url}')
    driver.get(url)
    time.sleep(6)
    driver.execute_script('window.scrollTo(0, 5000)')
    time.sleep(3)

    data = []
    halaman = 1

    while len(data) < jumlah_data and halaman <= batas_halaman:
        sup = BeautifulSoup(driver.page_source, 'html.parser')
        daftar_wadah = ambil_wadah_ulasan(sup)
        if not daftar_wadah:
            print(f'Halaman {halaman}: elemen ulasan tidak ditemukan')
            break

        jumlah_baru = 0
        for wadah in daftar_wadah:
            if len(data) >= jumlah_data:
                break

            ulasan = ambil_teks(wadah, [
                ('span', {'data-testid': 'lblItemUlasan'}),
                ('span', {'data-testid': 'lblReviewContent'}),
                ('p', {'data-testid': 'lblItemUlasan'}),
            ], 'Tidak ada ulasan')

            pelanggan = ambil_teks(wadah, [
                ('span', {'class': 'name'}),
                ('span', {'data-testid': 'lblReviewerName'}),
            ], 'Customer tidak ditemukan')

            elemen_rating = wadah.find(
                'div', attrs={'data-testid': 'icnStarRating'})
            label_rating = (elemen_rating['aria-label']
                            if elemen_rating else 'Tidak ada rating')
            rating = PETA_RATING.get(label_rating, 'Tidak ada rating')

            tanggal_relatif = ambil_tanggal(wadah)
            data.append({
                'platform': 'Tokopedia',
                'username': pelanggan,
                'rating': rating,
                'tanggal_raw': tanggal_relatif,
                'tanggal': konversi_tanggal(tanggal_relatif),
                'content': ulasan,
            })
            jumlah_baru += 1

        terisi = sum(1 for baris in data if baris['tanggal_raw'])
        print(f'Halaman {halaman:3d} | tambah {jumlah_baru:3d} | '
              f'total {len(data):4d} | tanggal terisi {terisi}')

        if len(data) >= jumlah_data:
            print('Jumlah data yang ditargetkan telah tercapai.')
            break

        # Berpindah ke halaman ulasan berikutnya
        try:
            tombol = driver.find_element(
                By.XPATH,
                "//button[contains(@aria-label,'Laman berikutnya') "
                "or contains(@aria-label,'next') "
                "or contains(@aria-label,'Next')]")
            if not tombol.is_enabled():
                print('Tombol halaman berikutnya nonaktif.')
                break
            driver.execute_script('arguments[0].click();', tombol)
            time.sleep(3)
            driver.execute_script('window.scrollTo(0, 2500)')
            time.sleep(2)
            halaman += 1
        except Exception:
            print(f'Halaman berikutnya tidak tersedia (halaman {halaman}).')
            break

    driver.quit()
    df = pd.DataFrame(data)

    terisi = df['tanggal_raw'].astype(bool).sum()
    print(f'\nLaporan kelengkapan tanggal')
    print(f'  Terisi : {terisi} ({terisi / len(df) * 100:.1f}%)')
    print(f'  Kosong : {len(df) - terisi} '
          f'({(len(df) - terisi) / len(df) * 100:.1f}%)')

    print(f'\nRingkasan data')
    print(f'  Total ulasan : {len(df)}')
    print(df['rating'].value_counts().sort_index())

    df.to_csv('tokopedia_reviews.csv', index=False)
    print('\nTersimpan ke tokopedia_reviews.csv')
    return df

df_tokopedia = scraping_ulasan(URL_ULASAN, JUMLAH_DATA)
display(df_tokopedia[['username', 'rating', 'tanggal_raw',
                      'tanggal', 'content']].head(10))

In [ ]:
# ---------------------------------------------------------------------
# BAGIAN 4 - PEMERIKSAAN KUALITAS DATA DAN UNDUH BERKAS
# ---------------------------------------------------------------------
from google.colab import files

BERKAS_HASIL = 'tokopedia_reviews.csv'
if not os.path.exists(BERKAS_HASIL):
    raise FileNotFoundError('Berkas hasil belum terbentuk. '
                            'Pastikan Bagian 3 telah selesai.')

df = pd.read_csv(BERKAS_HASIL)
print(f'Total baris        : {len(df)}')
print(f'Memuat teks ulasan : {(df["content"] != "Tidak ada ulasan").sum()}')
print(f'Tanggal terisi     : {df["tanggal_raw"].notna().sum()}')
print(f'Tanggal kosong     : {df["tanggal_raw"].isna().sum()}')

print('\nDistribusi tanggal relatif (sepuluh teratas)')
print(df['tanggal_raw'].value_counts().head(10))

print('\nDistribusi rating')
print(df['rating'].value_counts().sort_index())

print('\nPratinjau data')
display(df[['username', 'rating', 'tanggal_raw',
            'tanggal', 'content']].head(5))

files.download(BERKAS_HASIL)